### Similarity Test 

Using a defined embedding a list of sampled points, we use the vector value to determine the similarity between each pair of points. 

In [ ]:
# Import necessary libraries 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from itertools import combinations
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Parameters 

VECTOR_FILE_PATH = "../data/sample/poi_embeddings_cdmx.csv"
FIGURES_PATH = "../docs/resources/embedding_showcase/earth_embedings_sim_test_"

LANG = "esp"


In [ ]:
# Load embeddings CSV
df = pd.read_csv(VECTOR_FILE_PATH)

**Heatmap for Cosine Similarity**

In [ ]:
# Extract embedding matrix — A00–A63
band_cols         = [f'A{i:02d}' for i in range(64)]
labels            = df['label'].values
embeddings        = df[band_cols].values                  

# Cosine similarity
norms             = np.linalg.norm(embeddings, axis=1, keepdims=True)
embeddings_normed = embeddings / norms
cosine_sim        = embeddings_normed @ embeddings_normed.T

# Table
df_table = pd.DataFrame(
  np.round(cosine_sim, 4),
  index   = labels,
  columns = labels,
)
print(df_table.to_string())

### Heatmap 

n = len(labels)

if LANG == "esp":
  lab_01 = "Similaridad Coseno"
  lab_02 = "'Similaridad coseno — AlphaEarth Embeddings\nCDMX Puntos de Interés (2025)'"
else:
  lab_01 = "Cosine Similarity"
  lab_02 = "'Cosine Similarity — AlphaEarth Embeddings\nCDMX Points of Interest (2025)'"

# mask — hide upper triangle
mask_upper = np.triu(np.ones((n, n), dtype=bool), k=1)  # mask upper triangle 
diag_mask = ~np.eye(n, dtype=bool)                      # True everywhere except diagonal

fig, ax = plt.subplots(figsize=(11, 9))

# Base heatmap 
sns.heatmap(
  df_table,
  mask       = mask_upper,
  cmap       = 'YlGnBu',
  vmin       = 0.0,
  vmax       = 1.0,
  annot      = True,
  fmt        = '.3f',
  linewidths = 0.6,
  linecolor  = 'white',
  square     = True,
  ax         = ax,
  cbar_kws   = {'shrink': 0.7, 'label': lab_01},
  annot_kws  = {'size': 9},
)

# Gray diagonal overlay
sns.heatmap(
  pd.DataFrame(np.where(np.eye(n, dtype=bool), 1.0, np.nan), index=labels, columns=labels),
  mask       = diag_mask,
  cmap       = mcolors.ListedColormap(['#b0b0b0']),
  vmin       = 0,
  vmax       = 1,
  annot      = np.where(np.eye(n, dtype=bool), ['—']*n, ''),
  fmt        = '',
  linewidths = 0.6,
  linecolor  = 'white',
  square     = True,
  ax         = ax,
  cbar       = False,
  annot_kws  = {'size': 11, 'color': 'white', 'weight': 'bold'},
)

# Upper triangle — white fill
sns.heatmap(
  pd.DataFrame(np.zeros((n, n)), index=labels, columns=labels),
  mask       = ~mask_upper,
  cmap       = mcolors.ListedColormap(['white']),
  linewidths = 0.6,
  linecolor  = 'white',
  square     = True,
  ax         = ax,
  cbar       = False,
  annot      = False,
)

ax.set_title(
  lab_02,
  fontsize=13, pad=16, fontweight='bold',
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0,  fontsize=9)
plt.tight_layout()
if LANG == "esp":
  plt.savefig(FIGURES_PATH + 'cosine_similarity_heatmap_esp.png', dpi=150, bbox_inches='tight')
else:
  plt.savefig(FIGURES_PATH + 'cosine_similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

**PCA decomposition** (over the embedding vectors) 

In [ ]:
# Extract embedding matrix — A00–A63
band_cols         = [f'A{i:02d}' for i in range(64)]
labels            = df['label'].values
embeddings        = df[band_cols].values                  # shape (8, 64)

# Normalize (despite already been normalize by construction)
scaler            = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings)

# PCA — 2 principal components
pca               = PCA(n_components=2)
components        = pca.fit_transform(embeddings_scaled)  # shape (8, 2)
var_explained     = pca.explained_variance_ratio_

print(f'PC1 explained variance: {var_explained[0]:.2%}')
print(f'PC2 explained variance: {var_explained[1]:.2%}')
print(f'Total explained:        {sum(var_explained):.2%}')

# Plot 
fig, ax = plt.subplots(figsize=(10, 8))


for i, label in enumerate(labels):
    ax.scatter(
        components[i, 0], components[i, 1],
        color  = "steelblue",
        s      = 180,
        zorder = 3,
        edgecolors = 'white',
        linewidths = 0.8,
    )
    ax.annotate(
        label,
        xy         = (components[i, 0], components[i, 1]),
        xytext     = (8, 8),
        textcoords = 'offset points',
        fontsize   = 9,
        color      = "steelblue",
        fontweight = 'bold',
    )

# Origin crosshairs
ax.axhline(0, color='gray', linewidth=0.6, linestyle='--', alpha=0.5)
ax.axvline(0, color='gray', linewidth=0.6, linestyle='--', alpha=0.5)

ax.set_xlabel(f'PC1  ({var_explained[0]:.2%} variance explained)', fontsize=11)
ax.set_ylabel(f'PC2  ({var_explained[1]:.2%} variance explained)', fontsize=11)
ax.set_title(
    'PCA — AlphaEarth Embeddings (64D to 2D)\nCDMX Metro Area Points of Interest (2025)',
    fontsize=13, fontweight='bold', pad=14,
)
ax.set_facecolor('#f8f8f8')
ax.grid(True, linewidth=0.4, color='white')
plt.tight_layout()
plt.savefig(FIGURES_PATH + 'pca_embeddings_cdmx.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Extraer matriz de embeddings — A00–A63

band_cols         = [f'A{i:02d}' for i in range(64)]
labels            = df['label'].values
embeddings        = df[band_cols].values                  # forma (8, 64)

# Normalizar (aunque ya están normalizados por construcción)

scaler            = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings)

# PCA — 2 componentes principales

pca               = PCA(n_components=2)
components        = pca.fit_transform(embeddings_scaled)  # forma (8, 2)
var_explained     = pca.explained_variance_ratio_

print(f'Varianza explicada por PC1: {var_explained[0]:.2%}')
print(f'Varianza explicada por PC2: {var_explained[1]:.2%}')
print(f'Varianza total explicada:   {sum(var_explained):.2%}')

# Gráfico

fig, ax = plt.subplots(figsize=(10, 8))

for i, label in enumerate(labels):
    ax.scatter(
        components[i, 0], components[i, 1],
        color  = "steelblue",
        s      = 180,
        zorder = 3,
        edgecolors = 'white',
        linewidths = 0.8,
    )
    ax.annotate(
        label,
        xy         = (components[i, 0], components[i, 1]),
        xytext     = (8, 8),
        textcoords = 'offset points',
        fontsize   = 9,
        color      = "steelblue",
        fontweight = 'bold',
    )

# Líneas de referencia en el origen

ax.axhline(0, color='gray', linewidth=0.6, linestyle='--', alpha=0.5)
ax.axvline(0, color='gray', linewidth=0.6, linestyle='--', alpha=0.5)

ax.set_xlabel(
    f'PC1  ({var_explained[0]:.2%} de varianza explicada)',
    fontsize=11
)
ax.set_ylabel(
    f'PC2  ({var_explained[1]:.2%} de varianza explicada)',
    fontsize=11
)
ax.set_title(
    'PCA — Embeddings de AlphaEarth (64D a 2D)\n'
    'Puntos de interés del área metropolitana de CDMX (2025)',
    fontsize=13, fontweight='bold', pad=14,
)
ax.set_facecolor('#f8f8f8')
ax.grid(True, linewidth=0.4, color='white')
plt.tight_layout()
plt.savefig(FIGURES_PATH + 'pca_embeddings_cdmx_esp.png', dpi=150, bbox_inches='tight')
plt.show()